# Evaluate

In this notebook we evaluate the accuracy of the predicted alignments.

In [1]:
%matplotlib inline

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from matplotlib import cm
import seaborn as sns
from pathlib import Path
import glob
import os.path
import pandas as pd
import pickle
import re
from plotnine import *
from pandas.api.types import CategoricalDtype

In [3]:
DATASET = 'test' # 'test'
VERSION = 'full'

In [4]:
ANNOTATIONS_ROOT = Path('/home/ijain/ttmp/Chopin_Mazurkas/annotations_beat')
query_list = Path(f'cfg_files/queries.{DATASET}.{VERSION}')

### Evaluate hypothesis directory

In [5]:
def eval_dir(hypdir, querylist, annot_root_1, annot_root_2, hop_sec, savefile = None):
    allErrs = {}
    cnt = 0
    print(f'Processing {hypdir} ', end='')
    with open(querylist, 'r') as f:
        for line in f:
            parts = line.strip().split()
            assert len(parts) == 2
            basename = os.path.basename(parts[0]) + '__' + os.path.basename(parts[1])
            hypfile = hypdir + '/' + basename + '.pkl'
            if not os.path.exists(hypfile):
                print("X", end='')
                continue
            err = eval_file(hypfile, annot_root_1, annot_root_2, hop_sec)
            if err is not None:
                allErrs[basename] = err
            cnt += 1
            if cnt % 500 == 0:
                print(".", end='')
    print(' done')
    if savefile:
        pickle.dump(allErrs, open(savefile, 'wb'))
        
    return allErrs

In [6]:
def eval_file(hypfile, annot_root_1, annot_root_2, hop_sec):
    parts = os.path.basename(hypfile).split('__')
    assert len(parts) == 2
    piece = extractPieceName(parts[0])
    annotfile1 = (annot_root_1 / piece / parts[0]).with_suffix('.beat')
    annotfile2 = (annot_root_2 / piece / parts[1]).with_suffix('.beat')
    
    # if groundtruth annotation files are empty, skip this hypothesis file
    try:
        gt1, gt2 = getTimestamps(annotfile1, annotfile2)
        hypalign = loadAlignment(hypfile) # warping path in frames
    except:
        print(f'Skipping hypothesis file {hypfile}')
        return None

    if hypalign is None:
        err = np.array(np.ones(gt1.shape) * np.inf)
    else:
        pred2 = np.interp(gt1, hypalign[0,:]*hop_sec, hypalign[1,:]*hop_sec)
        err = pred2 - gt2
    return err

In [7]:
def extractPieceName(fullpath):
    basename = os.path.basename(fullpath) # e.g. Chopin_Op068No3_Sztompka-1959_pid9170b-21
    parts = basename.split('_')
    piece = '_'.join(parts[0:2]) # e.g. Chopin_Op068No3
    return piece

In [8]:
def getTimestamps(annotfile1, annotfile2):
    df1 = pd.read_csv(annotfile1, header=None, sep='\s+', skiprows=3) 
    df2 = pd.read_csv(annotfile2, header=None, sep='\s+', skiprows=3)

    df_merged = pd.merge(df1, df2, on=[2], how='inner')
    df_merged = df_merged[df_merged[2].astype(str).str.match(".*\d$")]

    return df_merged['0_x'], df_merged['0_y']

In [9]:
def loadAlignment(hypfile):
    with open(hypfile, 'rb') as f:
        d = pickle.load(f)
    return d

In [10]:
def eval_all_dirs(rootdir, querylist, hop_sec, outdir, benchmark, system, L=None):
    """Evaluate a system (including sparse_parflex with optional L) for a given benchmark.
    For sparse_parflex with a chunk length L, hypdir = rootdir/benchmark/sparse_parflex/L_<L>.
    For other systems, hypdir = rootdir/benchmark/system."""
    if not os.path.exists(outdir):
        os.mkdir(outdir)
    if benchmark == 'partialOverlap':
        annot_root_1 = Path(f'/home/ijain/ttmp/Chopin_Mazurkas_Benchmarks/partialStart/annotations_beat')
        annot_root_2 = Path(f'/home/ijain/ttmp/Chopin_Mazurkas_Benchmarks/partialEnd/annotations_beat')
    elif 'prepost' in benchmark:
        sec = benchmark.split('_')[-1]
        annot_root_1 = Path(f'/home/ijain/ttmp/Chopin_Mazurkas_Benchmarks/pre_{sec}/annotations_beat')
        annot_root_2 = Path(f'/home/ijain/ttmp/Chopin_Mazurkas_Benchmarks/post_{sec}/annotations_beat')
    elif benchmark == 'matching':
        annot_root_1 = Path(f'/home/ijain/ttmp/Chopin_Mazurkas_Modified/annotations_beat')
        annot_root_2 = Path(f'/home/ijain/ttmp/Chopin_Mazurkas_Modified/annotations_beat')
    else:
        annot_root_1 = Path(f'/home/ijain/ttmp/Chopin_Mazurkas_Benchmarks/{benchmark}/annotations_beat')
        annot_root_2 = Path(f'/home/ijain/ttmp/Chopin_Mazurkas_Modified/annotations_beat')
    if system == 'sparse_parflex' and L is not None:
        hypdir = f'{rootdir}/{benchmark}/sparse_parflex/L_{L}'
        system_label = f'sparse_parflex_L_{L}'
    else:
        hypdir = f'{rootdir}/{benchmark}/{system}'
        system_label = system
    if os.path.isdir(hypdir):
        outpath = outdir + '/' + f'{benchmark}'
        Path(outpath).mkdir(parents=True, exist_ok=True)
        savefile = outpath + '/' + f'{system_label}' + '.pkl'
        allErrs = eval_dir(hypdir, querylist, annot_root_1, annot_root_2, hop_sec, savefile=savefile)

**Evaluate on Data**

In [11]:
EXPERIMENTS_ROOT = f'experiments_{DATASET}/{VERSION}'
hop_sec = 512 * 1 / 22050
outdir = f'evaluations_{DATASET}/{VERSION}'
Path(outdir).mkdir(parents=True, exist_ok=True)

In [12]:
def eval_benchmark(experiments_root, hop_sec, outdir, benchmark, system, L=None):
    eval_all_dirs(experiments_root, query_list, hop_sec, outdir, benchmark, system, L)

In [13]:
# L values for sparse_parflex (saved under experiments_.../benchmark/sparse_parflex/L_<L>/)
from run_sparse_parflex_L_sweep_config import L_VALUES
# Systems to evaluate (as in 04_Align_Benchmarks.ipynb)
# (Parflex is not included as a separate base system here; each sparse_parflex L is treated as its own system.)
BASE_SYSTEMS = ['dtw1', 'dtw2', 'dtw3', 'subseqdtw1', 'subseqdtw2', 'subseqdtw3', 'nwtw', 'flexdtw']

# File labels for sparse_parflex with different L values
SPARSE_PARFLEX_FILE_LABELS = [f'sparse_parflex_L_{L}' for L in L_VALUES]

# Display labels for plotting
BASE_SYSTEMS_DISPLAY = ['DTW1', 'DTW2', 'DTW3', 'SubseqDTW1', 'SubseqDTW2', 'SubseqDTW3', 'NWTW', 'FlexDTW']
SPARSE_PARFLEX_DISPLAY_LABELS = [f'sparse_parflex L={L}' for L in L_VALUES]

SYSTEM_FILE_LABELS = BASE_SYSTEMS + SPARSE_PARFLEX_FILE_LABELS
SYSTEM_DISPLAY_LABELS = BASE_SYSTEMS_DISPLAY + SPARSE_PARFLEX_DISPLAY_LABELS
BENCHMARKS = ['Matching', 'Subseq_20', 'Subseq_30', 'Subseq_40', 'PartialStart', 'PartialEnd', 'PartialOverlap', 
              'Pre_5', 'Pre_10', 'Pre_20', 'Post_5', 'Post_10', 'Post_20', 'PrePost_5', 
              'PrePost_10', 'PrePost_20']

In [14]:
benchmarks = []
for benchmark in BENCHMARKS:
    if benchmark == 'PartialOverlap':
        benchmarks.append('partialOverlap')
    elif benchmark == 'PartialStart':
        benchmarks.append('partialStart')
    elif benchmark == 'PartialEnd':
        benchmarks.append('partialEnd')
    elif 'PrePost' in benchmark:
        sec = benchmark.split('_')[-1]
        benchmarks.append(f'prepost_{sec}')
    else:
        benchmarks.append(benchmark.lower())

In [15]:
def eval_all_benchmarks(experiments_root, hop_sec, outdir, benchmarks):
    for benchmark in benchmarks:
        # evaluate base systems (no L)
        for system in BASE_SYSTEMS:
            eval_benchmark(experiments_root, hop_sec, outdir, benchmark, system)
        # evaluate sparse_parflex for each L
        for L in L_VALUES:
            eval_benchmark(experiments_root, hop_sec, outdir, benchmark, 'sparse_parflex', L)          

In [16]:
eval_all_benchmarks(EXPERIMENTS_ROOT, hop_sec, outdir, benchmarks)

Processing experiments_test/full/matching/dtw1 ....... done
Processing experiments_test/full/matching/dtw2 ....... done
Processing experiments_test/full/matching/dtw3 ....... done
Processing experiments_test/full/matching/subseqdtw1 ....... done
Processing experiments_test/full/matching/subseqdtw2 ....... done
Processing experiments_test/full/matching/subseqdtw3 ....... done
Processing experiments_test/full/matching/nwtw ....... done
Processing experiments_test/full/matching/flexdtw ....... done
Processing experiments_test/full/matching/parflex ....... done
Processing experiments_test/full/matching/sparse_parflex/L_4000 ....... done
Processing experiments_test/full/matching/sparse_parflex/L_6000 ....... done
Processing experiments_test/full/subseq_20/dtw1 ....... done
Processing experiments_test/full/subseq_20/dtw2 ....... done
Processing experiments_test/full/subseq_20/dtw3 ....... done
Processing experiments_test/full/subseq_20/subseqdtw1 ....... done
Processing experiments_test/full

### Plot error vs tolerance

In [16]:
def calc_error_rates(errFile, maxTol):
    print(errFile)
    # read from file
    with open(errFile, 'rb') as f:
        allErrs = pickle.load(f)
    
    # collect all errors
    errsFlat = []
    # print(allErrs)
    for query in allErrs:
        # print(query)
        errs = np.array(allErrs[query])
        errsFlat.append(errs)
    # print(errsFlat)
    errsFlat = np.concatenate(errsFlat)
    
    # calculate error rates
    errRates = np.zeros(maxTol+1)
    for i in range(maxTol+1):
        errRates[i] = np.mean(np.abs(errsFlat) > i/1000)
    
    return errRates, errsFlat

In [17]:
def calc_error_rates_batch(indir, basenames, maxTol):
    errRates = np.zeros((len(basenames), maxTol+1))
    allErrVals = []
    print('Computing error rates ', end='')
    for i, basename in enumerate(basenames):
        errFile = indir + '/' + basename + '.pkl'
        errRates[i,:], errors = calc_error_rates(errFile, maxTol)
        allErrVals.append(errors)
        print('.', end='')
    print(' done')
    return errRates, allErrVals

In [18]:
def plot_multiple_roc(errRates, basenames):
    numSystems = errRates.shape[0]
    maxTol = errRates.shape[1] - 1
    for i in range(numSystems):
        plt.plot(np.arange(maxTol+1), errRates[i,:] * 100.0)
    plt.legend(basenames, bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)
    plt.xlabel('Error Tolerance (ms)')
    plt.ylabel('Error Rate (%)')
    plt.show()
    return

**Evaluate on Data**

In [20]:
EVAL_ROOT_DIR = f'evaluations_{DATASET}/{VERSION}'
toPlot = []

for benchmark in benchmarks:
    for system_label in SYSTEM_FILE_LABELS:
        toPlot.append(f'{benchmark}/{system_label}')
maxTol = 1000 # in msec

In [21]:
def get_errs(eval_root_dir, toPlot, maxTol):
    errRates, errVals = calc_error_rates_batch(EVAL_ROOT_DIR, toPlot, maxTol)
    return errRates, errVals

In [22]:
errRates, errVals = get_errs(EVAL_ROOT_DIR, toPlot, maxTol)

Computing error rates evaluations_test/full/matching/dtw1.pkl
.evaluations_test/full/matching/dtw2.pkl
.evaluations_test/full/matching/dtw3.pkl
.evaluations_test/full/matching/subseqdtw1.pkl
.evaluations_test/full/matching/subseqdtw2.pkl
.evaluations_test/full/matching/subseqdtw3.pkl
.evaluations_test/full/matching/nwtw.pkl
.evaluations_test/full/matching/flexdtw.pkl
.evaluations_test/full/matching/sparse_parflex_L_4000.pkl
.evaluations_test/full/matching/sparse_parflex_L_6000.pkl
.evaluations_test/full/subseq_20/dtw1.pkl
.evaluations_test/full/subseq_20/dtw2.pkl
.evaluations_test/full/subseq_20/dtw3.pkl
.evaluations_test/full/subseq_20/subseqdtw1.pkl
.evaluations_test/full/subseq_20/subseqdtw2.pkl
.evaluations_test/full/subseq_20/subseqdtw3.pkl
.evaluations_test/full/subseq_20/nwtw.pkl
.evaluations_test/full/subseq_20/flexdtw.pkl
.evaluations_test/full/subseq_20/sparse_parflex_L_4000.pkl
.evaluations_test/full/subseq_20/sparse_parflex_L_6000.pkl
.evaluations_test/full/subseq_30/dtw1.p

In [23]:
errRates.shape[0] == len(SYSTEM_FILE_LABELS) * len(BENCHMARKS)

True

In [24]:
np.save(f'evaluations_{DATASET}/{VERSION}_errRates', errRates)

### Make Plots (New)

In [25]:
def make_df(time, errRates):
    data = {}
    
    num_systems = len(SYSTEM_DISPLAY_LABELS)
    num_benchmarks = len(BENCHMARKS)
    
    data['Benchmark'] = []
    for benchmark in BENCHMARKS:
        data['Benchmark'] += [benchmark] * num_systems
    
    data['System'] = [label for label in SYSTEM_DISPLAY_LABELS] * num_benchmarks

    data['Error'] = []
    for i in range(num_benchmarks * num_systems):
        data['Error'].append(errRates[i][time]*100)
            
    df = pd.DataFrame.from_dict(data)
    benchmark_categories = CategoricalDtype(categories=BENCHMARKS, ordered=True)
    df.Benchmark = df.Benchmark.astype(benchmark_categories)
    system_categories = CategoricalDtype(categories=SYSTEM_DISPLAY_LABELS, ordered=True)
    df['System'] = df['System'].astype(system_categories)
    
    return df

In [26]:
errRates = np.load(f'evaluations_{DATASET}/{VERSION}_errRates.npy')

In [27]:
# Error rates at 200ms tolerance
ms200_df = make_df(200, errRates)

In [28]:
# Error rates at 100ms and 500ms tolerance (black horizontal bars on plot)
ms100_df = make_df(100, errRates)
ms500_df = make_df(500, errRates)

In [29]:
colors = [to_hex(cm.tab20(i % 20)) for i in range(len(SYSTEM_DISPLAY_LABELS))]


In [31]:
(ggplot(ms200_df, aes(x="System", y="Error", fill="System")) +
    geom_bar(width=0.7,
             position=position_dodge2(preserve='single', width=0.25),
             stat='identity') +
    scale_y_continuous(expand=(0, 0), limits=(0, 100)) +
    scale_fill_manual(values=colors) +
    geom_crossbar(aes(ymin="Error", ymax="Error"),
                  data=ms100_df, width=0.5, color="black") +
    geom_crossbar(aes(ymin="Error", ymax="Error"),
                  data=ms500_df, width=0.5, color="black") +
    facet_grid('. ~ Benchmark') +
    theme_bw() +
    labs(y="Error Rate (%)") +
    theme(
        dpi=600,
        figure_size=(15, 5),
        legend_position='bottom',
        legend_direction="horizontal",
        legend_box="horizontal",
        legend_background=element_blank(),
        legend_title=element_text(size=10),
        legend_text=element_text(size=8),
        strip_background=element_blank(),
        strip_text_x=element_text(size=9, weight='bold'),
        axis_text_x=element_blank(),
        axis_ticks_x=element_blank(),
        axis_text_y=element_text(size=6, colour='black'),
        axis_title_x=element_blank(),
        axis_title_y=element_text(size=10, margin={'r': 6.0})
    ) +
    guides(fill=guide_legend(
        nrow=1, byrow=True, title="",
        override_aes={'size': 0}
    ))
)


In [32]:
ms200_df.to_csv('evaluations.csv')

In [33]:
ms200_df.pivot_table(index='Benchmark', columns='System', values='Error')

/tmp/ipykernel_1447119/39499490.py:1: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior


System,DTW1,DTW2,DTW3,SubseqDTW1,SubseqDTW2,SubseqDTW3,NWTW,FlexDTW,sparse_parflex L=4000,sparse_parflex L=6000
Benchmark,,,,,,,,,,
Matching,7.831387,7.942210,6.694218,6.667253,7.166351,5.832426,4.815167,4.976265,5.353280,5.171341
Subseq_20,100.000000,100.000000,100.000000,10.135276,12.552880,10.332052,83.296069,8.252773,8.431855,8.276901
Subseq_30,100.000000,100.000000,100.000000,8.434363,10.361137,8.782951,69.082810,7.624171,7.803969,7.719207
Subseq_40,99.665437,99.660028,99.764697,7.825043,9.262555,7.543220,49.233237,7.456401,7.627875,7.482366
PartialStart,67.085189,67.055754,69.858879,7.550345,8.145709,6.662714,15.330893,5.177221,5.249142,5.177221
PartialEnd,72.746658,72.685576,74.997409,8.224177,8.717854,7.333078,20.186730,6.082242,6.379641,6.247586
PartialOverlap,100.000000,100.000000,100.000000,95.395646,92.753954,93.875391,88.128547,8.201135,8.324784,8.201135
Pre_5,11.868917,11.980826,10.934330,7.981028,8.310632,7.060170,5.837760,4.820797,5.239099,4.979129
Pre_10,17.253508,17.383493,17.083125,9.186054,9.278802,8.198229,5.881022,4.814969,5.149808,4.981697


In [96]:
pd.DataFrame(ms200_df.pivot(index='Benchmark', columns='L', values='Error'))

KeyError: 'L'

In [ ]:
import pandas as pd
ms200_df = pd.read_csv('evaluations.csv')

In [ ]:
ms500_df

,Benchmark,L,Error
0,Matching,L_100,60.236870
1,Matching,L_200,30.805353
2,Matching,L_300,20.952387
3,Matching,L_400,16.499280
4,Matching,L_500,13.451030
...,...,...,...
267,PrePost_20,L_4000,4.588754
268,PrePost_20,L_5000,4.558326
269,PrePost_20,L_6000,4.230917
270,PrePost_20,L_7000,4.082593


In [ ]:
ms100_df

,Benchmark,L,Error
0,Matching,L_100,68.664734
1,Matching,L_200,43.683149
2,Matching,L_300,34.847664
3,Matching,L_400,30.460373
4,Matching,L_500,27.772086
...,...,...,...
267,PrePost_20,L_4000,18.526680
268,PrePost_20,L_5000,18.477759
269,PrePost_20,L_6000,18.138792
270,PrePost_20,L_7000,17.963438


In [ ]:
x = ms200_df.pivot(index='Benchmark', columns='L', values='Error')

In [ ]:
# L sweep: pivot shows error rate per benchmark per L
x

L,L_100,L_1000,L_200,L_2000,L_300,L_3000,L_400,L_4000,L_500,L_5000,L_600,L_6000,L_700,L_7000,L_800,L_8000,L_900
Benchmark,,,,,,,,,,,,,,,,,
Matching,63.942434,13.017015,35.900800,10.427152,26.207798,9.631624,21.579899,9.256135,18.639991,9.153122,16.935066,8.859494,15.365600,8.609231,14.683567,8.463540,13.904637
PartialEnd,55.899099,13.881586,35.410329,11.307520,27.145031,10.253876,22.687230,9.475420,19.996322,9.131582,17.251320,8.961925,16.412968,8.878719,15.441349,8.752632,14.139858
PartialOverlap,36.744555,10.319173,23.042783,8.440845,18.507427,8.336196,15.182430,7.500250,13.798908,7.429590,12.382113,7.578782,11.757971,7.489875,11.328282,7.238358,10.695554
PartialStart,47.088705,9.827840,24.898243,8.016142,18.834336,7.892880,15.546266,7.226481,13.468424,7.228911,12.387968,7.260856,11.163937,7.130562,10.808211,6.948706,10.320804
Post_10,64.676267,13.044281,36.194882,10.397826,26.339138,9.587761,21.631799,9.203724,18.700763,9.066331,16.932196,8.791796,15.471295,8.505656,14.758877,8.357282,14.012267
Post_20,65.140481,13.229967,36.809529,10.466772,26.733096,9.609100,21.959868,9.216390,18.994703,9.068328,17.136101,8.780877,15.622103,8.464725,14.946996,8.325836,14.166817
Post_5,64.451897,12.990934,36.055182,10.388779,26.269693,9.586950,21.587811,9.209339,18.645170,9.122861,16.924709,8.800282,15.373524,8.557631,14.656925,8.414373,13.939141
PrePost_10,65.475663,12.982760,36.499117,10.470952,26.367652,9.541152,22.177811,9.287831,19.018537,9.121238,16.796426,8.781065,15.429429,8.529428,14.820648,8.387668,13.857841
PrePost_20,64.149373,13.108755,36.408381,10.153979,25.827923,9.275597,21.147800,8.942592,18.682201,8.904404,16.257048,8.540930,15.092370,8.353539,14.269678,8.223843,13.508933


In [ ]:
x

L,L_100,L_1000,L_200,L_2000,L_300,L_3000,L_400,L_4000,L_500,L_5000,L_600,L_6000,L_700,L_7000,L_800,L_8000,L_900
Benchmark,,,,,,,,,,,,,,,,,
Matching,63.942434,13.017015,35.900800,10.427152,26.207798,9.631624,21.579899,9.256135,18.639991,9.153122,16.935066,8.859494,15.365600,8.609231,14.683567,8.463540,13.904637
PartialEnd,55.899099,13.881586,35.410329,11.307520,27.145031,10.253876,22.687230,9.475420,19.996322,9.131582,17.251320,8.961925,16.412968,8.878719,15.441349,8.752632,14.139858
PartialOverlap,36.744555,10.319173,23.042783,8.440845,18.507427,8.336196,15.182430,7.500250,13.798908,7.429590,12.382113,7.578782,11.757971,7.489875,11.328282,7.238358,10.695554
PartialStart,47.088705,9.827840,24.898243,8.016142,18.834336,7.892880,15.546266,7.226481,13.468424,7.228911,12.387968,7.260856,11.163937,7.130562,10.808211,6.948706,10.320804
Post_10,64.676267,13.044281,36.194882,10.397826,26.339138,9.587761,21.631799,9.203724,18.700763,9.066331,16.932196,8.791796,15.471295,8.505656,14.758877,8.357282,14.012267
Post_20,65.140481,13.229967,36.809529,10.466772,26.733096,9.609100,21.959868,9.216390,18.994703,9.068328,17.136101,8.780877,15.622103,8.464725,14.946996,8.325836,14.166817
Post_5,64.451897,12.990934,36.055182,10.388779,26.269693,9.586950,21.587811,9.209339,18.645170,9.122861,16.924709,8.800282,15.373524,8.557631,14.656925,8.414373,13.939141
PrePost_10,65.475663,12.982760,36.499117,10.470952,26.367652,9.541152,22.177811,9.287831,19.018537,9.121238,16.796426,8.781065,15.429429,8.529428,14.820648,8.387668,13.857841
PrePost_20,64.149373,13.108755,36.408381,10.153979,25.827923,9.275597,21.147800,8.942592,18.682201,8.904404,16.257048,8.540930,15.092370,8.353539,14.269678,8.223843,13.508933
